# Сравнение полносвязной сети на NumPy и TensorFlow/Keras

Этот ноутбук показывает знакомые учебные примеры через TensorFlow/Keras и отдельные вспомогательные классы проекта. Основная библиотека `open_neuron` по-прежнему реализована на NumPy; TensorFlow здесь нужен только для сравнения.

Перед запуском установите optional dependency из каталога `library`: `pip install -e '.[tensorflow]'`. Ноутбук использует Keras-модель, собственный `CustomDense` и объект `Neuron` для просмотра параметров слоя. Это учебный пример, а не попытка повторить API Keras целиком.

В Keras модель обучается через `compile()` и `fit()`: оптимизатор, функция потерь и метрики задаются при компиляции; число эпох, размер батча, перемешивание и callback передаются в `fit()`. Параметра `learning_rate` у `fit()` нет — его задают в оптимизаторе.

In [1]:
from pathlib import Path
import sys

import numpy as np
import tensorflow as tf

# Найдём корень library, чтобы сравнение запускалось и из Jupyter, открытого workspace.
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "comparisons" / "tensorflow").is_dir():
        library_root = candidate
        break
    if (candidate / "library" / "comparisons" / "tensorflow").is_dir():
        library_root = candidate / "library"
        break
else:
    raise RuntimeError("Не найден каталог library/comparisons/tensorflow")

if str(library_root) not in sys.path:
    sys.path.insert(0, str(library_root))

from comparisons.tensorflow import CustomDense, Neuron, TrainingProgressCallback

np.set_printoptions(precision=4, suppress=True)
tf.keras.utils.set_random_seed(7)
print(f"TensorFlow: {tf.__version__}")

TensorFlow: 2.21.0


## 1. Регрессия: несколько примеров и отдельная тестовая часть

Пусть целевая величина задана линейной зависимостью от двух признаков. Мы выделяем обучающую, валидационную и тестовую части до обучения. Валидация помогает наблюдать за обучением, а тестовая часть остаётся для итоговой оценки.

Здесь один `CustomDense` с линейной активацией вычисляет `y = X @ W + b`. Для такой задачи скрытый слой не нужен: мы обучаем простую линейную модель и можем сравнить её с предсказанием базового уровня — средним значением обучающих целей.

In [2]:
x1 = np.linspace(-1.0, 1.0, 80, dtype=np.float32)
x2 = (0.7 * np.sin(2 * np.pi * x1)).astype(np.float32)
X = np.column_stack((x1, x2))
y = (0.8 * x1 - 0.5 * x2 + 0.3).reshape(-1, 1).astype(np.float32)

X_train, y_train = X[:56], y[:56]
X_validation, y_validation = X[56:68], y[56:68]
X_test, y_test = X[68:], y[68:]

regression_model = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    CustomDense(1, activation="linear", name="regression_output"),
])
regression_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
    loss="mse",
    metrics=["mae"],
)
regression_model.fit(
    X_train,
    y_train,
    validation_data=(X_validation, y_validation),
    epochs=40,
    batch_size=8,
    shuffle=True,
    verbose=0,
    callbacks=[TrainingProgressCallback(report_count=6)],
)

regression_predictions = regression_model.predict(X_test, verbose=0)
regression_mse = np.mean((regression_predictions - y_test) ** 2)
baseline_predictions = np.full_like(y_test, y_train.mean())
baseline_mse = np.mean((baseline_predictions - y_test) ** 2)
print(f"Тестовый MSE модели: {regression_mse:.6f}")
print(f"Тестовый MSE базового среднего: {baseline_mse:.6f}")
print("Первые предсказания:", regression_predictions[:5].ravel())

# Вызываем модель напрямую, чтобы сохранить последнее состояние для просмотра нейрона.
regression_model(X_test[:1], training=False)
regression_layer = regression_model.get_layer("regression_output")
print("Параметры выходного нейрона:")
print(regression_layer.neurons[0])

Обучение: эпох 40
Эпоха 1/40: loss=0.5962, mae=0.6442, val_loss=2.1745, val_mae=1.4167
Эпоха 6/40: loss=0.0017, mae=0.0337, val_loss=0.0007, val_mae=0.0237
Эпоха 12/40: loss=0.0001, mae=0.0068, val_loss=0.0000, val_mae=0.0050
Эпоха 18/40: loss=0.0000, mae=0.0010, val_loss=0.0000, val_mae=0.0024
Эпоха 24/40: loss=0.0000, mae=0.0002, val_loss=0.0000, val_mae=0.0002
Эпоха 30/40: loss=0.0000, mae=0.0000, val_loss=0.0000, val_mae=0.0000
Эпоха 36/40: loss=0.0000, mae=0.0000, val_loss=0.0000, val_mae=0.0000
Эпоха 40/40: loss=0.0000, mae=0.0000, val_loss=0.0000, val_mae=0.0000
Обучение завершено; итоговая потеря: 0.0000
Тестовый MSE модели: 0.000000
Тестовый MSE базового среднего: 1.523962
Первые предсказания: [1.2216 1.2474 1.2643 1.2725 1.2723]
Параметры выходного нейрона:
Neuron 1: inputs=[ 0.7215 -0.6888], weights=[ 0.8 -0.5], bias=0.3, z=[1.2216], activation=linear, a=[1.2216]


## 2. AND: один нейрон и вся таблица истинности

Для AND один сигмоидальный нейрон может разделить входы на два класса. Начальные веса и смещение указаны явно, чтобы было видно, как они меняются при обучении.

Мы обучаемся и показываем предсказания на всех четырёх строках таблицы AND. Это проверяет, научилась ли модель воспроизводить заданную таблицу. Поскольку все строки использовались при обучении, такая проверка сама по себе не измеряет качество на новых данных.

In [3]:
X_boolean = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y_and = np.array([[0], [0], [0], [1]], dtype=np.float32)

and_model = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    CustomDense(
        [Neuron(weights=[0.45, -0.12], bias=1.0)],
        activation="sigmoid",
        name="and_output",
    ),
])
and_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.7, momentum=0.3),
    loss="mse",
)
and_model.fit(
    X_boolean,
    y_and,
    epochs=100,
    batch_size=1,
    shuffle=False,
    verbose=0,
    callbacks=[TrainingProgressCallback(report_count=5)],
)

and_probabilities = and_model.predict(X_boolean, verbose=0).ravel()
and_classes = (and_probabilities >= 0.5).astype(int)
print("Вероятности:", and_probabilities)
print("Предсказанные классы:", and_classes)
print("Ожидаемые классы:    ", y_and.ravel().astype(int))
print("Совпали со всей таблицей:", np.array_equal(and_classes, y_and.ravel()))
and_model(X_boolean[-1:], training=False)
print("Обученный нейрон:")
print(and_model.get_layer("and_output").neurons[0])

Обучение: эпох 100
Эпоха 1/100: loss=0.4319
Эпоха 20/100: loss=0.0828
Эпоха 40/100: loss=0.0422
Эпоха 60/100: loss=0.0273
Эпоха 80/100: loss=0.0198
Эпоха 100/100: loss=0.0154
Обучение завершено; итоговая потеря: 0.0154
Вероятности: [0.0041 0.1311 0.1338 0.8492]
Предсказанные классы: [0 0 0 1]
Ожидаемые классы:     [0 0 0 1]
Совпали со всей таблицей: True
Обученный нейрон:
Neuron 1: inputs=[1. 1.], weights=[3.6191 3.5961], bias=-5.487, z=[1.7283], activation=sigmoid, a=[0.8492]


## 3. XOR: почему нужен скрытый слой

XOR нельзя разделить одной прямой в исходном двумерном пространстве. Скрытый слой из двух нейронов создаёт промежуточное представление, после чего выходной нейрон разделяет его.

И здесь все четыре возможных булевых входа участвуют в обучении и затем проверяются повторно. Верное совпадение с XOR-таблицей демонстрирует способность этой архитектуры представить таблицу; оно не заменяет независимую тестовую выборку и не является оценкой обобщения на широкий набор данных.

In [4]:
y_xor = np.array([[0], [1], [1], [0]], dtype=np.float32)

xor_model = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    CustomDense(
        [
            Neuron(weights=[-0.42, -0.37], bias=0.55),
            Neuron(weights=[0.36, 0.41], bias=-0.15),
        ],
        activation="sigmoid",
        name="xor_hidden",
    ),
    CustomDense(
        [Neuron(weights=[-0.7, 0.8], bias=-0.3)],
        activation="sigmoid",
        name="xor_output",
    ),
])
xor_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1),
    loss="binary_crossentropy",
)
xor_model.fit(
    X_boolean,
    y_xor,
    epochs=400,
    batch_size=4,
    shuffle=False,
    verbose=0,
    callbacks=[TrainingProgressCallback(report_count=5)],
)

xor_probabilities = xor_model.predict(X_boolean, verbose=0).ravel()
xor_classes = (xor_probabilities >= 0.5).astype(int)
print("Вероятности:", xor_probabilities)
print("Предсказанные классы:", xor_classes)
print("Ожидаемые классы:    ", y_xor.ravel().astype(int))
print("Совпали со всей таблицей:", np.array_equal(xor_classes, y_xor.ravel()))
xor_model(X_boolean[-1:], training=False)
print("Обученные нейроны скрытого слоя:")
for neuron in xor_model.get_layer("xor_hidden").neurons:
    print(neuron)
print("Обученный выходной нейрон:")
print(xor_model.get_layer("xor_output").neurons[0])

Обучение: эпох 400
Эпоха 1/400: loss=0.7009
Эпоха 80/400: loss=0.0689
Эпоха 160/400: loss=0.0136
Эпоха 240/400: loss=0.0071
Эпоха 320/400: loss=0.0045
Эпоха 400/400: loss=0.0032
Обучение завершено; итоговая потеря: 0.0032
Вероятности: [0.0038 0.9971 0.9971 0.003 ]
Предсказанные классы: [0 1 1 0]
Ожидаемые классы:     [0 1 1 0]
Совпали со всей таблицей: True
Обученные нейроны скрытого слоя:
Neuron 1: inputs=[1. 1.], weights=[6.6747 6.6751], bias=-10.0329, z=[3.3169], activation=sigmoid, a=[0.965]
Neuron 2: inputs=[1. 1.], weights=[9.3016 9.2959], bias=-4.3297, z=[14.2679], activation=sigmoid, a=[1.]
Обученный выходной нейрон:
Neuron 1: inputs=[0.965 1.   ], weights=[-12.5769  12.0543], bias=-5.7228, z=[-5.8053], activation=sigmoid, a=[0.003]


## Что сравнивать с `open_neuron`

- В обоих случаях полносвязный слой сначала считает взвешенные суммы входов с добавлением смещения, затем применяет активацию.
- В нашем NumPy-пакете вычисления и обучение реализованы учебным кодом проекта. Здесь Keras выполняет обучение через свою систему слоёв и `GradientTape`/градиенты; `CustomDense` задаёт вычисление слоя, а `Neuron` лишь показывает параметры и последнее состояние выхода.
- `Neuron` внутри `CustomDense` не является отдельной обучающей сущностью: матрицей весов и вектором смещений владеет слой Keras.
- Скорость и итог обучения зависят от инициализации, оптимизатора, шага обучения, функции потерь и числа эпох. Сравнивать реализации корректно при одинаковых данных и близких настройках, а не только по совпадению одного вывода.

См. [руководство Keras по созданию слоёв](https://keras.io/guides/making_new_layers_and_models_via_subclassing/) и [API обучения модели](https://keras.io/api/models/model_training_apis/).